## 1. Imports and Setup

In [1]:
import os, warnings
warnings.filterwarnings('ignore')

CFG = dict(
    SEED          = 42,
    N_FOLDS       = 2,
    DATA_DIR      = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR    = '/kaggle/working',
)

os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)

OPTS = ['A', 'B', 'C', 'D', 'E']

In [2]:
import re
import os, numpy as np, pandas as pd

train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
print(f"train: {train.shape} | test: {test.shape}")
y_idx = np.array([OPTS.index(a) for a in train['answer']])

train: (2000, 8) | test: (500, 7)


## 2. Text normalization

Prompts carry repetitive instruction boilerplate ("pick the best possible answer", "choose the correct option", ...). Stripping it lets the models focus on the actual question and, just as importantly, makes the lookup key robust to minor wording differences.

In [3]:
import re
from sklearn.model_selection import GroupKFold

_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    return _WS.sub(' ', s).strip()

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id

assert (train['fold'] >= 0).all()
_dup = train.groupby('prompt_n')['fold'].nunique()
assert _dup.max() == 1, "Leak: a prompt spans multiple folds"
print("Folds:\n", train['fold'].value_counts().sort_index().to_string())

Folds:
 fold
0    1000
1    1000


## 3. Leak-safe cross-validation

`GroupKFold` is grouped on the **normalized prompt** so paraphrase-duplicates can never land in different folds. The asserts above prove no prompt spans two folds — without this the CV score would be optimistically inflated.


## 4. The competition metric — MAP@3

Each row is scored on the rank of the correct option within the top-3 prediction (1.0 / 1.0 / 0.5 / 0.33 / 0 ...).

In [4]:
def map_at_3(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    y_idx = np.asarray(y_idx)
    order = (-scores_2d).argsort(axis=1)
    total = 0.0
    for i in range(len(scores_2d)):
        top3 = list(order[i, :3])
        if y_idx[i] in top3:
            total += 1.0 / (top3.index(y_idx[i]) + 1)
    return total / len(scores_2d)

def all_metrics(scores_2d, y_idx):
    scores_2d = np.asarray(scores_2d).reshape(-1, 5)
    pred = scores_2d.argmax(axis=1)
    return dict(map3=map_at_3(scores_2d, y_idx),
                acc=accuracy_score(y_idx, pred),
                f1=f1_score(y_idx, pred, average='macro'))

In [5]:
from sklearn.metrics import accuracy_score, f1_score

# sanity check
_t = np.array([[0.1,0.9,0.05,0.03,0.02],[0.5,0.2,0.1,0.1,0.1]]); _y = np.array([1,0])
print('metric sanity:', all_metrics(_t, _y))

metric sanity: {'map3': 1.0, 'acc': 1.0, 'f1': 1.0}
